# 章节实践

通过 07.01 到 07.03 的学习，我们已经掌握了基于指针的C语言算子开发常见的优化手段及其基本原理。为了巩固所学知识，现提供以下实践练习：

实现 Vector 算子 Gelu，算子文件名为 `gelu.asc`，核函数名为 `gelu_custom`，我们将为您创建如下目录及文件：

```c
└── src
    └── 07.04_simd_practice
        ├── gelu.h          // gelu算子Device侧实现
        ├── gelu.asc        // gelu算子Host侧实现
        └── CMakeLists.txt  // CMake配置
```

请补齐 kernel 侧代码，并完成 kernel 直调测试。

**计算公式**：

GELU 近似计算公式为：

$$
GELU(x) \approx 0.5 \cdot x \cdot \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \cdot \left(x + 0.044715 \cdot x^3\right)\right)\right)
$$

对公式进行简化，采用如下方式展开为具体的向量运算步骤：

$$
GELU(x) \approx \frac{x}{1 + e^{-1.595769 \cdot x - 0.071405 \cdot x^3}}
$$

要求：

- 输入输出 shape 均为 `[1, 805306368]`，输入输出类型均为 `float`。
- 使用基于指针的C语言编程完成开发。
- 请合理选择数据切分方式。
- NPU 架构使用 `dav-3510`。
- 已为您实现了数据准备、结果校验等基础功能。您可聚焦于算子实现。

请开始你的实践，体验从理解到创造的完整开发过程。

---

环境准备：


In [ ]:
!mkdir -p src/07.04_simd_practice

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---

gelu.asc 代码补全（需补全的位置已添加“TODO”注释）：


In [ ]:
%%writefile src/07.04_simd_practice/gelu.asc

#include <iostream>
#include <vector>
#include <iterator>
#include <random>
#include "acl/acl.h"
#include "gelu.h"

// ======================= 构造随机数据 =========================
float random_value()
{
    static std::random_device rd;
    static std::mt19937_64 gen(rd());
    static std::uniform_real_distribution<double> dist(0.0, 1.0); // 模板随机 value [0,1]
    return static_cast<float>(dist(gen));
}

void generate_test_data(std::vector<float>& values, uint32_t num)
{
    values.resize(num);

    for (uint32_t i = 0; i < num; ++i) {
        values[i] = random_value();
    }
}

float gelu_reference(float x)
{
    double x_fp64 = static_cast<double>(x);
    double exponent = static_cast<double>(COEFF_LINEAR) * x_fp64 +
        static_cast<double>(COEFF_CUBIC) * x_fp64 * x_fp64 * x_fp64;
    return static_cast<float>(x_fp64 / (1.0 + std::exp(exponent)));
}

// ======================= 构造golden数据 =========================
std::vector<float> gelu_reference(std::vector<float>& src, uint32_t total_len)
{
    // 支持 float 和 half 数据类型。half 类型先 cast 为 float 进行计算。
    std::vector<float> dst(src.size());
    for (uint32_t i = 0; i < total_len; ++i) {
        dst[i] = gelu_reference(src[i]);
    }
    return dst;
}

// ======================= 校验输出结果 =========================
uint32_t verify_result(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_func = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    std::cout << "Output size" << ": " << output.size() << std::endl;
    std::cout << "Golden size" << ": " << golden.size() << std::endl;
    print_func(output, "Output");
    print_func(golden, "Golden");
    constexpr float epsilon = 1e-4f;
    bool passed = true;
    uint32_t errCount = 0;
    for (size_t i = 0; i < golden.size(); i++) {
        if (std::abs(static_cast<float>(output[i]) - static_cast<float>(golden[i])) > epsilon) {
            passed = false;
            errCount++;
        }
    }
    if (passed) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed! errCount:" << errCount << std::endl;
        return 1;
    }
}

#define CHECK_ACL_RET(expr, src_ptr, dst_ptr, strem_var)                    \
    do {                                                                    \
        aclError ret = (expr);                                              \
        if (ret != ACL_SUCCESS) {                                           \
            post_proc(src_ptr, dst_ptr, strem_var);                         \
            std::cout << #expr << " failed, ret = " << ret << std::endl;    \
            return 1;                                                       \
        }                                                                   \
    } while (0)

void post_proc(uint8_t* src_device, uint8_t* dst_device, aclrtStream& stream) {
    if (src_device != nullptr) {
        aclrtFree(src_device);
    }

    if (dst_device != nullptr) {
        aclrtFree(dst_device);
    }


    if (stream != nullptr) {
        aclrtDestroyStream(stream);
    }

    aclrtResetDevice(0);
    aclFinalize();
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t total_len = 805306368;

    std::vector<float> src(total_len);
    generate_test_data(src, total_len);
    size_t input_size = total_len * sizeof(float);
    uint8_t* src_device = nullptr;
    uint8_t* dst_device = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    CHECK_ACL_RET(aclrtCreateStream(&stream), src_device, dst_device, stream);

    CHECK_ACL_RET(aclrtMalloc((void**)&src_device, input_size, ACL_MEM_MALLOC_HUGE_FIRST), src_device, dst_device, stream);
    CHECK_ACL_RET(aclrtMalloc((void**)&dst_device, input_size, ACL_MEM_MALLOC_HUGE_FIRST), src_device, dst_device, stream);

    CHECK_ACL_RET(aclrtMemcpy(src_device, input_size, src.data(), input_size, ACL_MEMCPY_HOST_TO_DEVICE), src_device, dst_device, stream);

    // TODO: 请补全 kernel 调用操作
    
    CHECK_ACL_RET(aclrtSynchronizeStream(stream), src_device, dst_device, stream);

    std::vector<float> output(total_len);
    CHECK_ACL_RET(aclrtMemcpy(output.data(), input_size, dst_device, input_size, ACL_MEMCPY_DEVICE_TO_HOST), src_device, dst_device, stream);

    std::vector<float> golden = gelu_reference(src, total_len);
    uint32_t result = verify_result(output, golden);

    post_proc(src_device, dst_device, stream);

    return result;
}

gelu.h 代码补全（需补全的位置已添加“TODO”注释）：

In [ ]:
%%writefile src/07.04_simd_practice/gelu.h

// TODO: 请根据您选用的API类型，引用对应的头文件。

__global__ __vector__ void gelu_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
{
    // TODO: 请补齐 kernel 侧实现。
}

生成 CMakeLists.txt 文件（直接执行，无需修改）

In [ ]:
%%writefile src/07.04_simd_practice/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo gelu.asc)

target_link_libraries(demo PRIVATE m)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

---

执行以下命令进行编译并验证结果：


In [ ]:
!cd src/07.04_simd_practice/ && mkdir -p build
!cd src/07.04_simd_practice/build && cmake .. && make -j
!cd src/07.04_simd_practice/build && ./demo

预期执行效果如下：

```text
[Success] Case accuracy is verification passed.
```

---

执行以下代码获取参考答案（参考答案选用C API实现）。

In [ ]:
!cat ./answer/07.04_simd_practice/gelu.asc

In [ ]:
!cat ./answer/07.04_simd_practice/gelu.h